# TRANSFORMACIÓN DE DATOS

**IMPORTANTE**: Recuerda hacer una copia de esta plantilla para no machacar la original.

## IMPORT PACKAGES

In [13]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
from category_encoders import TargetEncoder
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.preprocessing import Binarizer
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import QuantileTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

#Automcompletar rápido
%config IPCompleter.greedy=True

## IMPORT DATA

In [14]:
project_path = Path ('/Users/gloriadelriomarquez/Documents/Career/Projects/ml-lab/client_churn')
data_path = project_path / "02_data" / "train"

# Cargar datasets
train_df = pd.read_pickle(data_path / "train_clean.pickle")
cat = pd.read_pickle(data_path / "cat_clean.pickle")
num = pd.read_pickle(data_path / "num_clean.pickle")

# Separar variable objetivo
target = train_df["is_churn"].copy()

# Eliminar identificador y target de las features
cat = cat.drop(columns=["client_id", "is_churn"], errors="ignore")
num = num.drop(columns=["client_id", "is_churn"], errors="ignore")

# Verificar que los índices están alineados
assert train_df.index.equals(cat.index)
assert train_df.index.equals(num.index)

# Construir matriz de características
X = pd.concat([num, cat], axis=1)
y = target

# Comprobar dimensiones
print(f"Dataset original: {train_df.shape}")
print(f"\nVariables categóricas: {cat.shape}")
print(f"Variables numéricas: {num.shape}")
print(f"\nMatriz de características X: {X.shape}")
print(f"Variable objetivo y: {y.shape}")

assert len(X) == len(y)
assert X.index.equals(y.index)

Dataset original: (679672, 41)

Variables categóricas: (679672, 10)
Variables numéricas: (679672, 29)

Matriz de características X: (679672, 39)
Variable objetivo y: (679672,)


## COLUM NAMES

In [15]:
cat_features = list(cat.columns)
cat_features

['gender',
 'registered_via',
 'city',
 'last_day_any_cancel',
 'has_multi_non_cancelled_day',
 'has_multi_transaction_day',
 'last_day_all_auto_renew',
 'last_day_any_auto_renew',
 'has_mixed_cancel_day',
 'age_unknown']

In [16]:
num_features = list(num.columns)
num_features

['age',
 'auto_renew_rate',
 'avg_amount_paid',
 'avg_payment_per_30d',
 'avg_plan_days',
 'avg_transactions_per_day',
 'cancel_rate',
 'days_to_expiry',
 'days_to_max_expiry',
 'extra_transaction_records',
 'last_day_max_payment',
 'last_day_n_transactions',
 'max_plan_days',
 'max_transactions_same_day',
 'membership_age_days',
 'multi_transaction_day_rate',
 'n_cancellations',
 'n_distinct_plan_days',
 'n_multi_transaction_days',
 'n_non_cancelled_transactions',
 'n_payment_methods',
 'n_transaction_days',
 'n_transaction_records',
 'n_zero_list_price_transactions',
 'payment_amount_std',
 'recency_days',
 'total_amount_paid',
 'total_non_cancelled_amount',
 'transaction_span_days']

## CATEGORICAL FEATURES TRANSFORMATION

### One Hot Encoding

In [17]:
# Variables to apply One-Hot Encoding

# Variables para One-Hot Encoding
vars_ohe = [
    "gender",
    "registered_via",
    "city",
    "last_day_any_cancel",
    "has_multi_non_cancelled_day",
    "has_multi_transaction_day",
    "last_day_all_auto_renew",
    "last_day_any_auto_renew",
    "has_mixed_cancel_day"
]

# Variables binarias que conservamos sin transformar
vars_binary = [
    "age_unknown"
]

# Instantiate encoder
ohe = OneHotEncoder(drop= None, sparse_output=False,  handle_unknown='ignore')

# Fit and transform
cat_ohe = ohe.fit_transform(cat[vars_ohe])

# Save as DataFrame
cat_ohe = pd.DataFrame(cat_ohe, columns=ohe.get_feature_names_out())

## MERGE TRANSFORMED DATASETS

In [18]:
# Merge all dataframes
df = pd.concat([cat[vars_binary], cat_ohe ], axis = 1)

In [19]:
df.shape

(679672, 50)

In [20]:
df.head(2)

,age_unknown,gender_Unknown,gender_female,gender_male,registered_via_13.0,registered_via_3.0,registered_via_4.0,registered_via_7.0,registered_via_9.0,registered_via_Unknown,...,has_multi_transaction_day_2.0,last_day_all_auto_renew_0.0,last_day_all_auto_renew_1.0,last_day_all_auto_renew_2.0,last_day_any_auto_renew_0.0,last_day_any_auto_renew_1.0,last_day_any_auto_renew_2.0,has_mixed_cancel_day_0.0,has_mixed_cancel_day_1.0,has_mixed_cancel_day_2.0
0,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


## FEATURE SCALING

In [21]:
Descartamos StandarScaler ya que en la fase de feature_selection comprobamos que se obtiene una mayor capacidad predictiva a partir de 

SyntaxError: invalid syntax (3835381714.py, line 1)

### Robust Scaler

In [22]:
# Binary variables: only 0/1 values (ignoring NaN)
binary_cols = [col for col in num.columns
               if set(num[col].dropna().unique()) <= {0, 1}]

# Variables to rescale using robust standardization
vars_rs = [col for col in num.columns if col not in binary_cols]

In [23]:
vars_rs

['age',
 'auto_renew_rate',
 'avg_amount_paid',
 'avg_payment_per_30d',
 'avg_plan_days',
 'avg_transactions_per_day',
 'cancel_rate',
 'days_to_expiry',
 'days_to_max_expiry',
 'extra_transaction_records',
 'last_day_max_payment',
 'last_day_n_transactions',
 'max_plan_days',
 'max_transactions_same_day',
 'membership_age_days',
 'multi_transaction_day_rate',
 'n_cancellations',
 'n_distinct_plan_days',
 'n_multi_transaction_days',
 'n_non_cancelled_transactions',
 'n_payment_methods',
 'n_transaction_days',
 'n_transaction_records',
 'n_zero_list_price_transactions',
 'payment_amount_std',
 'recency_days',
 'total_amount_paid',
 'total_non_cancelled_amount',
 'transaction_span_days']

In [24]:
# Variables to rescale using robust standardization
vars_rs = num.columns

# Percentiles used for the range
p_min = 10.0
p_max = 90.0

rs = RobustScaler(quantile_range=(p_min, p_max))
df_rs = rs.fit_transform(num[vars_rs])

# Add suffixes to variable names
rs_names = [variable + '_rs' for variable in vars_rs]

# Save as DataFrame
df_rs = pd.DataFrame(df_rs, columns=rs_names)

## MERGE RESCALED DATASETS

In [25]:
# Create a list with the dataframes to include in the analytical table
to_include = [df, df_rs, target]

# Merge all dataframes into the analytical table
df_table = pd.concat(to_include, axis=1)

In [26]:
df_table.shape

(679672, 80)

## SAVE DATASET AFTER DATA TRANSFORMATION

In [27]:
# Define file name
df_table_path = project_path / "02_data" / "train" / 'modeling_dataset.pickle'
# Save file
df_table.to_pickle(df_table_path)